In [ ]:
import numpy as np
import pandas as pd
import re 
from datetime import datetime
from pathlib import Path
from scipy.stats import linregress

class Frontage:
    def __init__(self, input_file: str, assay_date: datetime):
        self.input_file = input_file
        self.filename = Path(input_file).parts[-1]
        assert self.filename.endswith(".xlsx"), "Incoming data must be an Excel file"
        self.assay_date = assay_date
    
    def __repr__(self):
        return f"Frontage assay handler built on file: '{self.filename}' on day: {self.assay_date}"

class FrontageStability(Frontage):
    def __init__(self, input_file, assay_date):
        super().__init__(input_file, assay_date)
        self.scraped_data = None 

        self.excel_file = pd.ExcelFile(input_file)
        self.sheet_names = self.excel_file.sheet_names
        
        self.target_sheet = self.check_sheet_names()

        self.data = self.read_data()
        self.check_columns()
        self.scraped_data_transformation()
        self.calculate_remaining()

    
    def __repr__(self):
        return super().__repr__()
    
    def read_data(self):
        """Read in data and store."""
        df = pd.read_excel(self.input_file, sheet_name=self.target_sheet, usecols=range(20))
        df.columns = [c.lower().replace(" ", "_") for c in df.columns]
        df = df.loc[~(df.isna().all(axis=1))]
        return df

    def check_sheet_names(self):
        """Make sure there is a sheet name titled 'Raw' within the file."""

        # looking for any sheet resembling "raw data"
        for sheet in self.sheet_names:
            if re.search(r"[Rr]+aw", sheet):
                print(f"Valid sheet name '{sheet}' identified!")
                return sheet
        raise ValueError("Looking for sheet named 'Raw' but none found")
    
    def check_columns(self):
        """Change column headers and check to see if mandatory columns are present."""
        df = self.data.copy()
        
        # look for column names
        #TODO: these columns are those used in the first data delivery - do they change?
        mandatory_columns = ["cmpd_id", "sample_id", "is_area", "peak_area"]
        for m in mandatory_columns:
            if m not in df.columns:
                raise ValueError(f"Column '{m}' not found in data sheet.")
            
        print("All needed column names found!")
        self.scraped_data = df[mandatory_columns]
            
        return 
    
    def scraped_data_transformation(self):
        """Rename scraped data columns, transform data to prepare for calculations."""
        df = self.scraped_data.copy()

        # rename columns to resemble those used in CDD
        df = df.rename(columns=dict(
            zip(
                ["cmpd_id", "sample_id", "is_area", "peak_area"],
                ["molecule_name", "time", "is_area", "cmpd_area"]
            )
        ))
        
        # log10 transform values
        for column in ["is_area", "cmpd_area"]:
            component, suffix = column.split("_")
            df.loc[:, f"{component}_log10_{suffix}"] = np.log10(df[column])
        
        self.scraped_data = df 
        return 
    
    def calculate_remaining(self):
        """Calculate percent remaining and apply natural log to data."""
        df = self.scraped_data.copy()

        df = df.set_index(["molecule_name", "time"]).sort_index()
        df = df.reset_index("time")

        # calculate is/cmpd ratio
        df.loc[:, "area_ratio"] = df.cmpd_area / df.is_area

        # for each molecule, use ratio to determine percent remaining
        for mol in df.index.unique():
            data = df.loc[mol, :]
            
            ref_val = data[data.time==0].area_ratio
            percent_remaining = data.area_ratio / ref_val
            
            # transform values using the natural log
            ln_percent_remaining = np.log(percent_remaining)


            # insert into dataframe
            df.loc[mol, "percent_remaining"] = percent_remaining
            df.loc[mol, "ln_percent_remaining"] = ln_percent_remaining
            
        self.scraped_data = df.reset_index()
        
        return 
    
    def calculate_regression(self):
        """Grab time and natural log data, filter, and perform linear regression."""
        

In [67]:
file = '/Users/delafield/Library/CloudStorage/GoogleDrive-delafield@calicolabs.com/My Drive/Projects/ADME_CRO/2025-11-10 Calico Liver Microsome Stability Result.xlsx'
assay_date = datetime(2025, 10, 10)
fs = FrontageStability(file, assay_date)
fs.scraped_data

Valid sheet name 'Raw Data' identified!
All needed column names found!


,molecule_name,time,is_area,cmpd_area,is_log10_area,cmpd_log10_area,area_ratio,percent_remaining,ln_percent_remaining
0,CP-8-176,0.0,170060.66,230658.22,5.230604,5.362969,1.356329,1.000000,0.000000
1,CP-8-176,15.0,173904.39,229494.75,5.240311,5.360773,1.319660,0.972965,-0.027407
2,CP-8-176,30.0,180017.25,220933.18,5.255314,5.344261,1.227289,0.904861,-0.099974
3,CP-8-176,60.0,182474.01,224136.48,5.261201,5.350513,1.228320,0.905621,-0.099134
4,CP-8-178,0.0,172240.98,80607.04,5.236136,4.906373,0.467990,1.000000,0.000000
5,CP-8-178,15.0,177343.41,80136.30,5.248815,4.903829,0.451871,0.965557,-0.035051
6,CP-8-178,30.0,168628.44,76392.21,5.226931,4.883049,0.453021,0.968014,-0.032508
7,CP-8-178,60.0,160564.46,70108.90,5.205649,4.845773,0.436640,0.933012,-0.069337
8,Verapamil,0.0,238568.60,557562.67,5.377613,5.746294,2.337117,1.000000,0.000000
9,Verapamil,15.0,212402.29,29555.33,5.327159,4.470636,0.139148,0.059538,-2.821136


In [42]:
d = fs.scraped_data.set_index(["cmpd_id", "sample_id"]).sort_index()
d.loc[:, "area_ratio"] = d.peak_area / d.is_area
d = d.reset_index("sample_id")
for idx in d.index.unique():
    data = d.loc[idx, :]
    ref_val = data.loc[data.sample_id==0, "area_ratio"]
    d.loc[idx, "perc_remaining"] = data.area_ratio / ref_val
    log_reg = np.log(data.area_ratio / ref_val)
    d.loc[idx, "ln_perc_remainging"] = log_reg

    m, b, r, _, _ = linregress(data.sample_id.to_numpy(), log_reg)
    print(linregress(data.sample_id.to_numpy(), log_reg))
    d.loc[idx, "regression_slope"] = m 
    d.loc[idx, "half_life"] = -0.693/m 
d

LinregressResult(slope=-0.0017332613251500185, intercept=-0.011130900520868035, rvalue=-0.8738019750257805, pvalue=0.1261980249742195, stderr=0.0006820624677341736, intercept_stderr=0.023442021653101695)
LinregressResult(slope=-0.00105026946532003, intercept=-0.006654464621830086, rvalue=-0.9496899575497022, pvalue=0.050310042450297805, stderr=0.0002449142277088631, intercept_stderr=0.008417534904356372)
LinregressResult(slope=-0.07025065766322872, intercept=-0.9737763779184987, rvalue=-0.8924014091509991, pvalue=0.10759859084900092, stderr=0.02511800093820536, intercept_stderr=0.863288554539734)


,sample_id,is_area,peak_area,area_ratio,perc_remaining,ln_perc_remainging,regression_slope,half_life
cmpd_id,,,,,,,,
CP-8-176,0.0,170060.66,230658.22,1.356329,1.000000,0.000000,-0.001733,399.824302
CP-8-176,15.0,173904.39,229494.75,1.319660,0.972965,-0.027407,-0.001733,399.824302
CP-8-176,30.0,180017.25,220933.18,1.227289,0.904861,-0.099974,-0.001733,399.824302
CP-8-176,60.0,182474.01,224136.48,1.228320,0.905621,-0.099134,-0.001733,399.824302
CP-8-178,0.0,172240.98,80607.04,0.467990,1.000000,0.000000,-0.001050,659.830665
CP-8-178,15.0,177343.41,80136.30,0.451871,0.965557,-0.035051,-0.001050,659.830665
CP-8-178,30.0,168628.44,76392.21,0.453021,0.968014,-0.032508,-0.001050,659.830665
CP-8-178,60.0,160564.46,70108.90,0.436640,0.933012,-0.069337,-0.001050,659.830665
Verapamil,0.0,238568.60,557562.67,2.337117,1.000000,0.000000,-0.070251,9.864676
